# Masterclass: Prompt Engineering & Prompt Management in LangChain

This notebook is a comprehensive guide to **Prompt Engineering and Prompt Management**. It covers prompt anatomy, shot-based prompting strategies (Zero-Shot, One-Shot, Few-Shot), LangChain template types (`PromptTemplate` vs. `ChatPromptTemplate`), dynamic Jinja2 logic, externalizing prompts to JSON files, and centralized prompt management via **LangSmith Hub**.

---

## 1. Fundamentals of Prompt Construction

A **prompt** is the complete input given to an LLM to direct its behavior and specify output format.

### Key Components of an Effective Prompt:

| Component | Description | Example |
|---|---|---|
| **Instruction** | The core task command | *"Summarize the following customer feedback."* |
| **Question** | Specific inquiry to be answered | *"What are the main issues reported by customers?"* |
| **Context** | Background info or retrieved RAG documents | *"Feedback collected from food delivery app users."* |
| **Examples** | In-context demonstration pairs | *Input: "Delivery late" → Output: "Delivery Issue"* |
| **Constraints** | Strict boundaries and negative constraints | *"Use only supplied context. Under 100 words."* |
| **Output Format** | Structuring instructions | *"Return as: 1. Issue  2. Frequency  3. Explanation"* |

## 2. Shot-Based Prompting Strategies

Shot-based prompting refers to providing **in-context examples** inside the prompt to guide the model's pattern recognition.

```text
Zero-Shot  ──► 0 Examples  ──► Rely solely on model pre-trained knowledge
One-Shot   ──► 1 Example   ──► Demonstrate task pattern once
Few-Shot   ──► N Examples  ──► Provide multiple diverse examples for complex tasks
```

### Strategy Comparison:

- **Zero-Shot Prompting:**
  ```text
  Classify sentiment: "The service was excellent."
  ```

- **One-Shot Prompting:**
  ```text
  Example: "The product is amazing." → Positive
  Now classify: "The service was terrible."
  ```

- **Few-Shot Prompting:**
  ```text
  Example 1: "The product is amazing." → Positive
  Example 2: "The service was terrible." → Negative
  Example 3: "The food was okay." → Neutral
  Now classify: "The delivery was very fast."
  ```

## 3. LangChain Prompt Templates (`PromptTemplate` vs. `ChatPromptTemplate`)
LangChain provides two primary prompt template abstractions: `PromptTemplate` for single-string text prompts and `ChatPromptTemplate` for multi-role chat models.

### 3.1 Text-Based `PromptTemplate`
Construct a single string template with variable placeholders (`{topic}`, `{audience}`, `{word_limit}`) and chain it with `ChatGoogleGenerativeAI` using LCEL (`prompt | model`).

In [1]:
# Import PromptTemplate and ChatGoogleGenerativeAI from LangChain
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
# Create a PromptTemplate with variable placeholders
prompt = PromptTemplate.from_template(
    """
You are an AI instructor.

Explain {topic} to {audience}.

Requirements:
- Use simple English
- Include one practical example
- Keep the answer under {word_limit} words
"""
)

In [3]:
# Initialize ChatGoogleGenerativeAI model (gemini-3.1-flash-lite)
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite"
)

In [4]:
# Build LCEL chain combining prompt template and chat model
chain = prompt | model

In [5]:
# Invoke chain with dictionary of template variables
response = chain.invoke(
    {
        "topic": "Vector Database",
        "audience": "beginner developers",
        "word_limit": 200
    }
)

In [6]:
# Display model response content
print(response.content)

[{'type': 'text', 'text': 'A **vector database** is a specialized tool designed to store and search data based on its **meaning** rather than exact keywords.\n\nTraditional databases (like SQL) look for matches like "Is the word \'cat\' in this sentence?" Vector databases use **embeddings**—long lists of numbers (vectors) that represent the semantic essence of data. They map items into a multi-dimensional space where similar things are placed close together.\n\n### Practical Example: A Recommendation Engine\nImagine you are building a movie app. If a user searches for "a thrilling space adventure," a traditional database might fail if the description doesn\'t contain those exact words. \n\nA vector database, however, understands that "thrilling space adventure" is semantically similar to "Star Wars" or "Interstellar" because their vectors are positioned nearby in its mathematical space. It retrieves these movies instantly, even if the keywords don\'t match perfectly.\n\n**In short:** W

### 3.2 Multi-Role `ChatPromptTemplate`
Structure prompts into distinct system and human role messages (`('system', ...)`, `('human', ...)`). System messages establish persona/rules, while human messages contain user input.

In [7]:
# Import ChatPromptTemplate for role-based chat messaging
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

In [8]:
# Define ChatPromptTemplate with System and Human roles
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an experienced AI instructor. Use simple English."
        ),
        (
            "human",
            "Explain {topic} to {audience}. Include one example."
        )
    ]
)

In [9]:
# Initialize ChatGoogleGenerativeAI model
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [10]:
# Assemble LCEL chat chain
chain = prompt | model

In [11]:
# Invoke chat chain for beginner audience
response = chain.invoke(
    {
        "topic": "Vector Database",
        "audience": "beginner developers"
    }
)

In [12]:
# Print output for beginner audience query
print(response.content)

[{'type': 'text', 'text': 'To understand a **Vector Database**, you first need to understand how computers "read" data.\n\n### The Problem: Computers don\'t understand "meaning"\nTraditional databases (like SQL) are great at matching exact keywords. If you search for "Apple," they look for that specific word. But they don\'t know that "Apple" is a fruit, or that it is similar to "Pear."\n\n### The Solution: Vectors\nIn AI, we turn data (text, images, audio) into **Vectors**. A vector is just a long list of numbers. \n\nThink of it like a **coordinate on a map**. \n* Imagine a 3D map where "Fruit" is in one corner and "Cars" is in another. \n* A "Pear" would be a set of numbers that places it very close to "Apple." \n* A "Toyota" would be a set of numbers that places it very close to "Honda."\n\n**A Vector Database is a storage system designed to find data based on these "coordinates" (meaning) rather than exact keywords.**\n\n---\n\n### How it works:\n1. **Embedding:** You take your da

### Summary: `PromptTemplate` vs. `ChatPromptTemplate`

| Feature | `PromptTemplate` | `ChatPromptTemplate` |
|---|---|---|
| **Output Type** | Single plain string | List of structured role messages (`SystemMessage`, `HumanMessage`, `AIMessage`) |
| **Target Model** | Completion LLMs | Chat / Instruct Models (`ChatGoogleGenerativeAI`, `ChatGoogleGenerativeAI`) |
| **Role Distinction** | None (monolithic text) | Explicit System, Human, and AI message roles |

## 4. Dynamic Templating with Jinja2
Jinja2 templates allow you to add dynamic logic inside prompts—including conditional logic (`{% if %}` / `{% elif %}`) and loops (`{% for %}`). Set `template_format="jinja2"` when initializing templates.

### 4.1 Jinja2 in `PromptTemplate`
Use Jinja2 conditionals (`{% if include_example %}`) and level checks (`{% if level == 'beginner' %}`) inside single-text templates.

In [14]:
# Import PromptTemplate for Jinja2 template formatting
from langchain_core.prompts import PromptTemplate

In [15]:
# Define string template containing Jinja2 conditionals
template = """
You are an AI instructor.

Explain {{ topic }} to {{ audience }}.

{% if include_example %}
Include one practical example.
{% endif %}

{% if level == "beginner" %}
Use very simple English and avoid complex terminology.
{% elif level == "advanced" %}
Include technical details and architecture.
{% else %}
Use moderate technical depth.
{% endif %}
"""

In [16]:
# Instantiate PromptTemplate with template_format='jinja2'
prompt = PromptTemplate.from_template(
    template,
    template_format="jinja2"
)

In [17]:
# Inspect prompt object structure
prompt

PromptTemplate(input_variables=['audience', 'include_example', 'level', 'topic'], input_types={}, partial_variables={}, template='\nYou are an AI instructor.\n\nExplain {{ topic }} to {{ audience }}.\n\n{% if include_example %}\nInclude one practical example.\n{% endif %}\n\n{% if level == "beginner" %}\nUse very simple English and avoid complex terminology.\n{% elif level == "advanced" %}\nInclude technical details and architecture.\n{% else %}\nUse moderate technical depth.\n{% endif %}\n', template_format='jinja2')

In [18]:
# Invoke Jinja2 prompt with variable values
formatted_prompt = prompt.invoke(
    {
        "topic": "RAG",
        "audience": "Python developers",
        "include_example": True,
        "level": "beginner"
    }
)

In [19]:
# Render formatted prompt string to verify Jinja2 evaluation
print(formatted_prompt.to_string())


You are an AI instructor.

Explain RAG to Python developers.


Include one practical example.



Use very simple English and avoid complex terminology.



### 4.2 Customer Support Dynamic Prompting
Dynamically alter prompt instructions based on user tiers (e.g. `user_type == 'premium'`).

In [20]:
# Import PromptTemplate for Jinja2 customer support template
from langchain_core.prompts import PromptTemplate

In [21]:
# Define customer support template with user_type conditional logic
template = """
You are a customer support assistant.

Customer Query:
{{ query }}

{% if user_type == "premium" %}
Provide a detailed response and mention priority support.
{% else %}
Provide a concise response.
{% endif %}
"""

In [22]:
# Instantiate Jinja2 customer support PromptTemplate
prompt = PromptTemplate.from_template(
    template,
    template_format="jinja2"
)

In [25]:
# Interactive input for query and user type
describe_problem = input("Enter customer query: ")
difficulty_level = input("Enter difficulty level (easy/medium/hard): ")

In [26]:
# Format prompt using user inputs
result = prompt.invoke(
    {
        "query": describe_problem,
        "user_type": difficulty_level
    }
)

In [27]:
# Print formatted customer support prompt
print(result.to_string())


You are a customer support assistant.

Customer Query:
My payment failed.


Provide a detailed response and mention priority support.



### Jinja2 Templating Key Takeaway
Jinja2 prompting enables **dynamic prompt generation** at runtime. Using conditionals (`{% if %}`) and loops (`{% for %}`), you can adjust prompt complexity, rules, and output formatting based on user variables without writing separate template strings.

### 4.3 Jinja2 in `ChatPromptTemplate`
Apply Jinja2 conditional formatting across both System and Human messages in a `ChatPromptTemplate`.

In [28]:
# Import ChatPromptTemplate for Jinja2 multi-role chat templates
from langchain_core.prompts import ChatPromptTemplate

In [30]:
# Define ChatPromptTemplate using Jinja2 conditionals in System and Human messages
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI instructor.

{% if level == "beginner" %}
Use very simple English and avoid complex terminology.
{% elif level == "advanced" %}
Include technical details and architecture.
{% else %}
Use moderate technical depth.
{% endif %}
"""
        ),
        (
            "human",
            """
Explain {{ topic }} to {{ audience }}.

{% if include_example %}
Include one practical example.
{% endif %}
"""
        )
    ],
    template_format="jinja2"
)

In [31]:
# Invoke ChatPromptTemplate with variables

formatted_prompt = prompt.invoke(
    {
        "topic": "RAG",
        "audience": "Python developers",
        "include_example": True,
        "level": "beginner"
    }
)

### LCEL Execution Workflow

```text
User Question ("Explain RAG?")
           ↓
    Prompt Template
           ↓
      LLM Model
           ↓
    Output Parser
           ↓
    Final Response
```


In [32]:
# Inspect generated system and human chat messages
for message in formatted_prompt.to_messages():
    print(message.type.upper())
    print(message.content)

SYSTEM

You are an AI instructor.


Use very simple English and avoid complex terminology.

HUMAN

Explain RAG to Python developers.


Include one practical example.



### 4.4 Dynamic Loops with Jinja2 (`{% for %}`)
Iterate over lists (e.g. `{% for rule in rules %}`) to dynamically inject enterprise rules into system messages, and conditionally toggle response format (JSON vs. Markdown).

In [33]:
# Define ChatPromptTemplate with Jinja2 loops ({% for rule in rules %}) and output format check
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an enterprise AI assistant.

Follow these rules:

{% for rule in rules %}
- {{ rule }}
{% endfor %}
"""
        ),
        (
            "human",
            """
Question:
{{ question }}

{% if output_format == "json" %}
Return the response in JSON format.
{% else %}
Return the response in Markdown.
{% endif %}
"""
        )
    ],
    template_format="jinja2"
)

In [34]:
# Format prompt with rule list and JSON output format preference
result = prompt.invoke(
    {
        "rules": [
            "Do not fabricate information",
            "Keep the answer concise",
            "Use only the supplied context"
        ],
        "question": "What is RAG?",
        "output_format": "json"
    }
)

In [35]:
# Print evaluate system and human message content
for message in result.to_messages():
    print(message.type, ":", message.content)

system : 
You are an enterprise AI assistant.

Follow these rules:


- Do not fabricate information

- Keep the answer concise

- Use only the supplied context

human : 
Question:
What is RAG?


Return the response in JSON format.



### Comparison: Standard Formatter vs. Jinja2 Templating

```text
PromptTemplate + Standard Formatter
└── Simple {variable} substitution only

PromptTemplate / ChatPromptTemplate + Jinja2
├── Variable substitution ({{ variable }})
├── Conditional logic ({% if condition %})
└── Iterative loops ({% for item in list %})
```

### Architecture of JSON Prompt Management

```text
prompt.json
  ├── rag_prompt
  ├── summarization_prompt
  ├── classification_prompt
  └── code_review_prompt
            ↓
     load_prompt(name)
            ↓
    ChatPromptTemplate
            ↓
  LCEL Chain (prompt | model)
            ↓
      Final Response
```


### 5.1 Loading Simple String Prompts from JSON
Read prompt strings from a local JSON file and format them using Python dictionary values.

In [37]:
### Architecture of JSON Prompt Management

```text
prompt.json
  ├── rag_prompt
  ├── summarization_prompt
  ├── classification_prompt
  └── code_review_prompt
            ↓
     load_prompt(name)
            ↓
    ChatPromptTemplate
            ↓
  LCEL Chain (prompt | model)
            ↓
      Final Response
```


In [38]:
# Inspect loaded prompts dictionary
prompts

{'rag_prompt': 'Answer the question using only the provided context.\n\nContext: {context}\n\nQuestion: {question}',
 'summary_prompt': 'Summarize the following text in 3 bullet points.\n\nText: {text}',
 'classification_prompt': 'Classify the following text as Positive, Negative, or Neutral.\n\nText: {text}',
 'code_review_prompt': 'Review the following Python code and identify any bugs.\n\nCode: {code}'}

In [39]:
# Select rag_prompt configuration
prompt = prompts["rag_prompt"]

In [40]:
# Inspect rag_prompt dictionary
prompt

'Answer the question using only the provided context.\n\nContext: {context}\n\nQuestion: {question}'

In [41]:
# Format prompt string using loaded template and dictionary values
final_prompt = prompt.format(
    context="Employees receive 24 paid leaves every year.",
    question="How many paid leaves do employees receive?"
)

In [42]:
# Inspect formatted prompt string
final_prompt

'Answer the question using only the provided context.\n\nContext: Employees receive 24 paid leaves every year.\n\nQuestion: How many paid leaves do employees receive?'

In [43]:
# Display formatted prompt text
print(final_prompt)

Answer the question using only the provided context.

Context: Employees receive 24 paid leaves every year.

Question: How many paid leaves do employees receive?


### 5.2 Dynamic `ChatPromptTemplate` Loader Function
Define a modular helper function `load_prompt(prompt_name)` that parses JSON configuration files and instantiates `ChatPromptTemplate` objects dynamically.

In [44]:
# Import JSON module and ChatPromptTemplate
import json
from langchain_core.prompts import ChatPromptTemplate

In [45]:
# Define load_prompt helper to build ChatPromptTemplate dynamically from JSON config
def load_prompt(prompt_name):
    
    with open("prompt.json", "r", encoding="utf-8") as file:
        prompts = json.load(file)

    if prompt_name not in prompts:
        raise ValueError(
            f"Prompt '{prompt_name}' not found."
        )

    config = prompts[prompt_name]

    messages = [
        (message["role"], message["template"])
        for message in config["messages"]
    ]

    prompt = ChatPromptTemplate.from_messages(
        messages,
        template_format=config["template_format"]
    )

    return prompt

In [46]:
# Load rag_prompt using helper function
prompt = load_prompt("rag_prompt")

In [47]:
# Inspect loaded rag_prompt template object
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template="You are a RAG assistant. Answer only from the provided context. If the answer is not available, say 'I do not have enough information.'", template_format='jinja2'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Context:\n{{ context }}\n\nQuestion:\n{{ question }}', template_format='jinja2'), additional_kwargs={})])

In [48]:
# Invoke loaded rag_prompt with context and question
result = prompt.invoke(
    {
        "context": """
        Employees receive 24 paid leaves every year.
        Maximum 10 unused leaves can be carried forward.
        """,

        "question": "How many paid leaves are available?"
    }
)

In [49]:
# Print role messages generated from JSON-loaded prompt
for message in result.to_messages():
    print(message.type.upper())
    print(message.content)

SYSTEM
You are a RAG assistant. Answer only from the provided context. If the answer is not available, say 'I do not have enough information.'
HUMAN
Context:

        Employees receive 24 paid leaves every year.
        Maximum 10 unused leaves can be carried forward.
        

Question:
How many paid leaves are available?


In [50]:
# Load summarization_prompt using helper function
prompt = load_prompt("summarization_prompt")

In [51]:
# Inspect summarization_prompt object
prompt

ChatPromptTemplate(input_variables=['num_points', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a professional text summarizer.', template_format='jinja2'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['num_points', 'text'], input_types={}, partial_variables={}, template='Summarize the following text in {{ num_points }} bullet points:\n\n{{ text }}', template_format='jinja2'), additional_kwargs={})])

In [52]:
# Invoke summarization_prompt with variables
result = prompt.invoke(
    {
        "num_points": 3,
        "text": "How many paid leaves are available?"
    }
)

In [53]:
# Display role messages for summarization prompt
for message in result.to_messages():
    print(message.type.upper())
    print(message.content)

SYSTEM
You are a professional text summarizer.
HUMAN
Summarize the following text in 3 bullet points:

How many paid leaves are available?


### 5.3 Executing JSON-Configured Prompts with LLMs
Integrate loaded JSON prompts directly into LCEL chains (`prompt | model`) for automated inference.

In [54]:
# Build and execute LCEL chain using JSON-loaded prompt and ChatGoogleGenerativeAI model
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite"
)

prompt = load_prompt(
    "rag_prompt"
)

chain = prompt | model

response = chain.invoke(
    {
        "context": """
        Employees receive 24 paid leaves annually.
        """,

        "question":
        "How many paid leaves do employees receive?"
    }
)

print(response.content)

[{'type': 'text', 'text': 'Employees receive 24 paid leaves annually.', 'extras': {'signature': 'EjQKMgERTTIPruf9znBWPDAwAIm6eAox3ZZ9TZYZpwxWrvTudcAS1oX9Vlb8fuN7tX2j4gYK'}}]


### Architecture of JSON Prompt Management

```text
prompt.json
  ├── rag_prompt
  ├── summarization_prompt
  ├── classification_prompt
  └── code_review_prompt
            ↓
     load_prompt(name)
            ↓
    ChatPromptTemplate
            ↓
  LCEL Chain (prompt | model)
            ↓
      Final Response
```


## 6. Centralized Prompt Management via LangSmith Hub
For production teams, **LangSmith Hub** provides centralized prompt management, versioning, collaboration, and deployment without modifying codebase files.

### 6.1 Pushing Prompts to LangSmith Hub
Upload and version-control a prompt template in LangSmith Hub using `client.push_prompt()`.

In [ ]:
# Push prompt template to central LangSmith Hub repository
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate

client = Client()

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple English."
)

client.push_prompt(
    "ai-teaching-prompt",
    object=prompt
)

### 6.2 Pulling Prompts from LangSmith Hub
Fetch the latest or specific version of a prompt from LangSmith Hub at runtime using `client.pull_prompt()`.

In [ ]:
# Pull prompt template from LangSmith Hub and execute at runtime
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "ai-teaching-prompt"
)

result = prompt.invoke({
    "topic": "RAG"
})

print(result)